In [ ]:
# 00_era2_daily_correlation.ipynb -- Era 2 (2023-Oct 2024) daily corridor/cross-border vs frequency-stress correlation
# Re-download if runtime reset:
# !kaggle datasets download -d halcyonvector/india-power-grid-nldc-daily-psp-reports -p data --unzip

import numpy as np
import pandas as pd

df = pd.read_csv("data/study1_daily.csv", parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)

# Era 2 window: IR-line corridor cols are populated from 2023-01-01, cross-border cols
# from 2023-07-06 (both verified in Phase 3's audit) -- Era 2 ends where Era 3's live
# SCADA data begins (study2_scada starts 2024-11-04), so this window uses everything
# up to end-Oct 2024, the last full month before that handoff.
era2 = df[(df["date"] >= "2023-01-01") & (df["date"] <= "2024-10-31")].copy()

IR_COLS = [c for c in df.columns if c.startswith("ir_") and c.endswith("_net_mu")]
XB_COLS = [c for c in df.columns if c.startswith("xb_net_")]

# Stress proxy: % of the day spent outside the 49.9-50.05 Hz normal band (below 49.9 +
# above 50.05), from the existing freq_pct_* columns. Higher = more frequency instability.
era2["stress_pct"] = era2["freq_pct_below_499"] + era2["freq_pct_above_5005"]

print("Era 2 window:", era2.shape, era2["date"].min().date(), "->", era2["date"].max().date())

# --- IR-line (inter-regional) corridor congestion vs same-day stress ---
ir_sub = era2.dropna(subset=IR_COLS + ["stress_pct"])
print(f"\nIR-line rows usable: {len(ir_sub)} of {len(era2)}")
ir_abs_sum = ir_sub[IR_COLS].abs().sum(axis=1)
print("corr(sum|ir_net|, same-day stress_pct):", round(np.corrcoef(ir_abs_sum, ir_sub["stress_pct"])[0, 1], 3))
for c in IR_COLS:
    print(" ", c, round(np.corrcoef(ir_sub[c].abs(), ir_sub["stress_pct"])[0, 1], 3))

# --- Lagged version: does today's corridor flow predict TOMORROW's stress? ---
ir_lag = ir_sub.copy()
ir_lag["ir_abs_sum"] = ir_abs_sum
ir_lag = ir_lag.sort_values("date")
ir_lag["stress_pct_next"] = ir_lag["stress_pct"].shift(-1)
ir_lag_valid = ir_lag.dropna(subset=["stress_pct_next"])
print("\ncorr(ir_abs_sum today, stress_pct tomorrow):",
      round(np.corrcoef(ir_lag_valid["ir_abs_sum"], ir_lag_valid["stress_pct_next"])[0, 1], 3))

# --- Cross-border exchange vs same-day stress (from 2023-07-06 onward only) ---
xb_win = era2[era2["date"] >= "2023-07-06"].copy()
xb_sub = xb_win.dropna(subset=XB_COLS + ["stress_pct"])
print(f"\nCross-border rows usable: {len(xb_sub)} of {len(xb_win)}")
print("corr(sum|xb_net|, same-day stress_pct):",
      round(np.corrcoef(xb_sub[XB_COLS].abs().sum(axis=1), xb_sub["stress_pct"])[0, 1], 3))
for c in XB_COLS:
    v = xb_sub[c].abs()
    corr = np.corrcoef(v, xb_sub["stress_pct"])[0, 1] if v.std() > 0 else float("nan")
    print(" ", c, round(corr, 3) if corr == corr else "nan (zero variance -- no exchange recorded in this window)")

# --- Findings (verified 2026-07-11) ---
# IR-line corridor congestion has a moderate NEGATIVE correlation with same-day frequency
# stress (~-0.40 for the summed absolute net flow across all 7 corridors), strongest for
# WR<->NR (-0.42) and NER<->NR (-0.37). The lagged (today's flow vs tomorrow's stress)
# version is similar or slightly stronger (~-0.43). Negative direction makes physical
# sense: corridors move power specifically to relieve regional imbalance, so higher
# corridor utilization coincides with LOWER frequency instability, not higher -- i.e.
# corridors are evidence of the grid actively correcting stress, not causing it.
#
# Cross-border exchange shows close to no linear correlation with stress (~-0.01 to
# -0.17 depending on country; Myanmar's column is constant zero in this window --
# essentially no exchange recorded, consistent with it not being meaningfully
# grid-connected in the data). Bangladesh is the only country with even a mild signal
# (-0.17). This is a real, negative finding, not a null result to paper over: at DAILY
# resolution, cross-border exchange volume alone does not explain frequency stress in
# this window -- if it matters, it likely shows up at finer (SCADA) resolution or via a
# different framing (e.g. exchange volatility rather than level), which Era 3's live
# model (Phase 4 Era 3) is positioned to test.
#
# Feature-design takeaway for Era 3: IR-line net-flow magnitude (especially WR-NR and
# NER-NR) is the more promising corridor signal to carry into the SCADA-resolution
# classifier; cross-border columns are included there too but expectations should be
# calibrated low based on this daily-resolution pre-check.
